### Import the necessary database

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [3]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [4]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [10]:
ds

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164, lat: 90, lon: 180)>
 dask.array<open_dataset-df1d26385c17deb67f2ced416cec5bf3trend, shape=(164, 90, 180), dtype=float64, chunksize=(164, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163, lat: 90, lon: 180)>
 dask.array<open_dataset-a13757215e76170d6f332760606c3821trend, shape=(163, 90, 180), dtype=float64, chunksize=(163, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162, lat: 90, lon: 180)>
 dask.array<open_d

In [6]:
# flip the longitude to -180 to 180
ds_adj = {}
for key in ds.keys():
    ds_adj[key] = preprosess.adjust_longitude(ds[key],ds[key].lon)

In [7]:
ds_adj

{'ICV_trend_10yr': (array([[[-0.00876186, -0.0087619 , -0.00877973, ..., -0.00876184,
           -0.00876181, -0.00876182],
          [-0.00841684, -0.00842867, -0.00844424, ..., -0.00839057,
           -0.00839713, -0.00840502],
          [-0.00705928, -0.00710348, -0.00715207, ..., -0.00694285,
           -0.00697782, -0.00701509],
          ...,
          [-0.05359884, -0.05381129, -0.05397888, ..., -0.05292523,
           -0.05314104, -0.05338639],
          [-0.05438904, -0.0545001 , -0.05462539, ..., -0.05402935,
           -0.0541402 , -0.05427799],
          [-0.0546163 , -0.05461719, -0.05480954, ..., -0.05440526,
           -0.05440612, -0.05461542]],
  
         [[-0.01181036, -0.01181041, -0.01183444, ..., -0.01181033,
           -0.01181029, -0.01181031],
          [-0.0113453 , -0.01136124, -0.01138223, ..., -0.01130988,
           -0.01131873, -0.01132936],
          [-0.00951541, -0.00957498, -0.00964048, ..., -0.00935847,
           -0.0094056 , -0.00945584],
         

In [11]:
ds_temp = {}
for key in ds.keys():
    ds_temp[key] = xr.DataArray(
        ds_adj[key][0],
        coords={'segment': ds[key].segment, 'lat': ds[key].lat, 'lon': ds_adj[key][1]},
        dims=('segment', 'lat', 'lon')
    )

In [12]:
ds_temp

{'ICV_trend_10yr': <xarray.DataArray (segment: 164, lat: 90, lon: 180)>
 array([[[-0.00876186, -0.0087619 , -0.00877973, ..., -0.00876184,
          -0.00876181, -0.00876182],
         [-0.00841684, -0.00842867, -0.00844424, ..., -0.00839057,
          -0.00839713, -0.00840502],
         [-0.00705928, -0.00710348, -0.00715207, ..., -0.00694285,
          -0.00697782, -0.00701509],
         ...,
         [-0.05359884, -0.05381129, -0.05397888, ..., -0.05292523,
          -0.05314104, -0.05338639],
         [-0.05438904, -0.0545001 , -0.05462539, ..., -0.05402935,
          -0.0541402 , -0.05427799],
         [-0.0546163 , -0.05461719, -0.05480954, ..., -0.05440526,
          -0.05440612, -0.05461542]],
 
        [[-0.01181036, -0.01181041, -0.01183444, ..., -0.01181033,
          -0.01181029, -0.01181031],
         [-0.0113453 , -0.01136124, -0.01138223, ..., -0.01130988,
          -0.01131873, -0.01132936],
         [-0.00951541, -0.00957498, -0.00964048, ..., -0.00935847,
          -0

In [13]:
lat = ds_temp["ICV_trend_10yr"].lat
lon = ds_temp["ICV_trend_10yr"].lon
# Extratropical South Pacific region
# Arctic region 
lat1 = 66.5
lat2 = 90
lon1 = -180
lon2 = 180
# select the region

ds_Arctic_masked = {}
for key in ds.keys():
    ds_Arctic_masked[key] = data_process.selreg(ds_temp[key],lat, lon, lat1, lat2, lon1, lon2)

In [15]:
ds_Arctic_masked

{'ICV_trend_10yr': (<xarray.DataArray (segment: 164, lat: 12, lon: 180)>
  array([[[-2.13361212e-01, -2.17383544e-01, -1.04635720e-01, ...,
            1.01539858e-01,  1.40596326e-01, -2.09338880e-01],
          [-1.27324720e-01, -2.69713578e-01, -5.45191440e-02, ...,
           -4.37348777e-01, -4.47271262e-01, -1.16669318e-01],
          [-4.69799011e-01, -4.41568469e-01, -9.50729534e-02, ...,
           -4.82438960e-01, -5.10777552e-01, -4.88166209e-01],
          ...,
          [-5.35988402e-02, -5.38112949e-02, -5.39788783e-02, ...,
           -5.29252273e-02, -5.31410449e-02, -5.33863854e-02],
          [-5.43890443e-02, -5.45001002e-02, -5.46253932e-02, ...,
           -5.40293478e-02, -5.41402040e-02, -5.42779885e-02],
          [-5.46163025e-02, -5.46171853e-02, -5.48095376e-02, ...,
           -5.44052568e-02, -5.44061220e-02, -5.46154197e-02]],
  
         [[-3.24404422e-01, -3.02017032e-01, -4.09156618e-01, ...,
           -1.98116282e-01, -1.64451333e-01, -3.96482922e-01]

In [16]:
# calculate the regional mean
ds_Arctic_mean = {}
for key in ds_Arctic_masked.keys():
    ds_Arctic_mean[key] = data_process.calc_weighted_mean(ds_Arctic_masked[key][0])

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [17]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [18]:
# calculate the regional mean's percentile
# 5%---[0]
unforced_trend_arctic_lower_percentile = {}

# 95%---[1]
unforced_trend_arctic_upper_percentile = {}

for key in ds_Arctic_masked.keys():
    unforced_trend_arctic_lower_percentile[key], unforced_trend_arctic_upper_percentile[key] = calc_percentile(ds_Arctic_mean[key], 5)
    

In [19]:
unforced_trend_arctic_lower_percentile

{'ICV_trend_10yr': -0.5967382897828863,
 'ICV_trend_11yr': -0.5325754580030518,
 'ICV_trend_12yr': -0.5425473302088591,
 'ICV_trend_13yr': -0.519870592008294,
 'ICV_trend_14yr': -0.49122676808030835,
 'ICV_trend_15yr': -0.4958644574971119,
 'ICV_trend_16yr': -0.45230965580366717,
 'ICV_trend_17yr': -0.4081157477063236,
 'ICV_trend_18yr': -0.37808179677048187,
 'ICV_trend_19yr': -0.3595880809993561,
 'ICV_trend_20yr': -0.3376335764085967,
 'ICV_trend_21yr': -0.30689534493388543,
 'ICV_trend_22yr': -0.31158920215663,
 'ICV_trend_23yr': -0.31216782981007507,
 'ICV_trend_24yr': -0.32311802824517805,
 'ICV_trend_25yr': -0.3226992068845506,
 'ICV_trend_26yr': -0.3304934118121043,
 'ICV_trend_27yr': -0.3155936784828476,
 'ICV_trend_28yr': -0.31240466740574213,
 'ICV_trend_29yr': -0.3004219292750461,
 'ICV_trend_30yr': -0.2995828548122973,
 'ICV_trend_31yr': -0.29710639434963626,
 'ICV_trend_32yr': -0.2854854169455309,
 'ICV_trend_33yr': -0.2819375626442371,
 'ICV_trend_34yr': -0.2737672993772

In [20]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_arctic_lower_percentile.keys())
unforced_trend_Arctic_lower_percentile_da = xr.DataArray(
	list(unforced_trend_arctic_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_Arctic_upper_percentile_da = xr.DataArray(
	list(unforced_trend_arctic_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [21]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_Arctic_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_Arctic_trend_lower_percentile.nc')
unforced_trend_Arctic_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_Arctic_trend_upper_percentile.nc')
